In [1]:
# conda install -c conda-forge openbabel -y

In [2]:
import os
import glob
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
# 1. route dir
input_dir = os.path.join("ProAffinity-GNN", "data", "temp_pdb") # cached
output_dir = os.path.join("ProAffinity-GNN", "data", "pdbqt")
os.makedirs(output_dir, exist_ok=True)

# 2. Get pdbs
pdb_files = glob.glob(os.path.join(input_dir, "*.pdb"))
total = len(pdb_files)

# Define single task
def convert_pdb(pdb_path):
    basename = os.path.basename(pdb_path).replace(".pdb", "")
    out_path = os.path.join(output_dir, f"{basename}_atom_processed.pdbqt")
    
    # Skip typecasted
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return True

    # Calling OpenBabel ：-ipdb (input) -opdbqt (output) -p 7.4 (proton under pH 7.4 )    
    cmd = f"obabel -ipdb {pdb_path} -opdbqt -O {out_path} -p 7.4"
    try:
        subprocess.run(cmd, shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return os.path.exists(out_path)
    except:
        return False

# 3. Enable multi-core parallel processing
if __name__ == '__main__':
    print(f"Typecasting {total} pdbs to pdbqts via multithreading...")

    success_count = 0

    with ThreadPoolExecutor() as executor:
        # Multithread
        futures = {executor.submit(convert_pdb, path): path for path in pdb_files}
        
        # Status
        for i, future in enumerate(as_completed(futures), 1):
            if future.result():
                success_count += 1
            if i % 50 == 0:
                print(f"Running... : {i} / {total} ...")

print(f"\n Number of successful typecasting to pdbqt:{success_count}")
print(f"Folder saved in : {output_dir}")

Typecasting 1270 pdbs to pdbqts via multithreading...
Running... : 50 / 1270 ...
Running... : 100 / 1270 ...
Running... : 150 / 1270 ...
Running... : 200 / 1270 ...
Running... : 250 / 1270 ...
Running... : 300 / 1270 ...
Running... : 350 / 1270 ...
Running... : 400 / 1270 ...
Running... : 450 / 1270 ...
Running... : 500 / 1270 ...
Running... : 550 / 1270 ...
Running... : 600 / 1270 ...
Running... : 650 / 1270 ...
Running... : 700 / 1270 ...
Running... : 750 / 1270 ...
Running... : 800 / 1270 ...
Running... : 850 / 1270 ...
Running... : 900 / 1270 ...
Running... : 950 / 1270 ...
Running... : 1000 / 1270 ...
Running... : 1050 / 1270 ...
Running... : 1100 / 1270 ...
Running... : 1150 / 1270 ...
Running... : 1200 / 1270 ...
Running... : 1250 / 1270 ...


In [3]:
import os
import glob

out_files = glob.glob("ProAffinity-GNN/data/pdbqt/*_atom_processed.pdbqt")
valid_files = [f for f in out_files if os.path.getsize(f) > 0]
len(valid_files)

1270